# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

## Data Loading and Inspection

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df=pd.read_csv("data/AviationData.csv", encoding="latin-1",low_memory=False)
df.head()
               

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [4]:
print(df["Aircraft.Category"].value_counts(dropna=False))
print(df["Amateur.Built"].value_counts(dropna=False))

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64
Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [5]:
df["Event.Date"]=pd.to_datetime(df["Event.Date"],errors="coerce") #it converts the date
#to make the model work by standerdizing
df["Make"]=df["Make"].str.strip().str.upper()
df["Model"]=df["Model"].str.strip().str.upper()
known=df.dropna(subset=["Aircraft.Category","Make","Model"])
lookup=(known.groupby(["Make","Model"])["Aircraft.Category"]
    .agg(lambda s: s.mode().iloc[0]))
missing=df["Aircraft.Category"].isna()
keys=list(zip(df.loc[missing, "Make"],df.loc[missing, "Model"]))
df.loc[missing,"Aircraft.Category"]=[lookup.get(k,np.nan) for k in keys]

print("still missing category:",df["Aircraft.Category"].isna().sum())
    

still missing category: 11909


In [6]:
df=df[df["Aircraft.Category"]=="Airplane"]
df=df[df["Amateur.Built"]=="No"]
df=df[df["Event.Date"]>="1983-01-01"]
df.shape

(61384, 31)

#my assumptions
In the Aircraft category some values were absent so i removed them by using the make/Model which is the most common category thereby remaining with unkown 11,909 rows which was also excluded since it cannot be confirmed as aeroplanes
Events that occurred  before 1983 were also removed due to the forty years maximum make/model lifetime.


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [7]:
inj = ["Total.Fatal.Injuries", "Total.Serious.Injuries",
       "Total.Minor.Injuries", "Total.Uninjured"]
print(df[inj].isna().sum())

# Assumption, a missing injury count means none was recorded so fill with a 0
df[inj] = df[inj].fillna(0)

# Estimated number of people on board
df["Total.Occupants"] = df[inj].sum(axis=1)

# Drop rows with no occupant information, since we can't compute a rate
df = df[df["Total.Occupants"] > 0].copy()

# Derived measures
df["FatalSerious.Count"] = df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
df["FatalSerious.Rate"] = df["FatalSerious.Count"] / df["Total.Occupants"]
df["FatalSerious.Any"] = (df["FatalSerious.Count"] > 0).astype(int)

print(df.shape)
df[["Total.Occupants", "FatalSerious.Rate", "FatalSerious.Any"]].describe()

Total.Fatal.Injuries      7875
Total.Serious.Injuries    8652
Total.Minor.Injuries      8180
Total.Uninjured           3618
dtype: int64
(60312, 35)


,Total.Occupants,FatalSerious.Rate,FatalSerious.Any
count,60312.000000,60312.000000,60312.000000
mean,6.409056,0.269143,0.306805
std,28.535025,0.428012,0.461171
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,0.666667,1.000000
max,588.000000,1.000000,1.000000


Assumptions:
Total.Occupants=Fatal+serious=minor+uninjored
rows with 0 occupants was dropped since rate cannot be calculated
also the missing injury counts was set  to 0 assuming 0 was recorded

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [8]:
print(df["Aircraft.damage"].value_counts(dropna=False))

Aircraft.damage
Substantial    45141
Destroyed      11954
NaN             1587
Minor           1556
Unknown           74
Name: count, dtype: int64


In [9]:
# Cleaning: treat NaN and "Unknown" as missing damage info and drop them
df["Aircraft.damage"] = df["Aircraft.damage"].replace("Unknown", np.nan)
df = df.dropna(subset=["Aircraft.damage"]).copy()

# Derived column: 1 if the aircraft was destroyed, else 0
df["Destroyed"] = (df["Aircraft.damage"] == "Destroyed").astype(int)

print(df.shape)
print(df["Destroyed"].value_counts())
print("Destroyed rate:", df["Destroyed"].mean())

(58651, 36)
Destroyed
0    46697
1    11954
Name: count, dtype: int64
Destroyed rate: 0.20381579171710626


Aircraft.Damage Assumptions
Destroyed is if the Aircraft damage equals to Destroyed
overall destroyed rate is almost 20%

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [10]:
print("Unique makes:", df["Make"].nunique())
print("Missing:", df["Make"].isna().sum())
print(df["Make"].value_counts().head(40))

Unique makes: 1054
Missing: 1
Make
CESSNA                            24805
PIPER                             13533
BEECH                              4490
MOONEY                             1184
BOEING                             1005
GRUMMAN                             964
BELLANCA                            915
AIR TRACTOR                         624
AERONCA                             532
MAULE                               495
CHAMPION                            474
STINSON                             392
LUSCOMBE                            364
AERO COMMANDER                      330
TAYLORCRAFT                         318
NORTH AMERICAN                      294
DE HAVILLAND                        279
ROCKWELL                            248
AIR TRACTOR INC                     218
CIRRUS DESIGN CORP                  217
MCDONNELL DOUGLAS                   207
GRUMMAN AMERICAN                    184
AYRES                               184
CIRRUS                              139
ERCOU

In [11]:
for name in ["CESSNA", "PIPER", "BEECH", "BOEING", "AIR TRACTOR",
             "CIRRUS", "MOONEY", "GRUMMAN", "DE HAVILLAND", "ROCKWELL"]:
    print(name)
    print(df[df["Make"].str.contains(name, na=False)]["Make"].value_counts().head(8))
    print("-----")

CESSNA
Make
CESSNA                     24805
CESSNA AIRCRAFT CO            24
CESSNA AIRCRAFT COMPANY       11
CESSNA AIRCRAFT               10
CESSNA ECTOR                   6
CESSNA/AIR REPAIR INC          2
REIMS-CESSNA                   1
CESSNA/WEAVER                  1
Name: count, dtype: int64
-----
PIPER
Make
PIPER                         13533
PIPER AIRCRAFT INC               25
PIPER AIRCRAFT CORPORATION       10
PIPER/CUB CRAFTERS                7
PIPER AIRCRAFT                    6
NEW PIPER AIRCRAFT INC            6
PIPER AIRCRAFT, INC.              3
NEW PIPER                         2
Name: count, dtype: int64
-----
BEECH
Make
BEECH                            4490
BEECHCRAFT                         17
HAWKER BEECHCRAFT CORP             13
HAWKER BEECHCRAFT                  13
HAWKER BEECHCRAFT CORPORATION       7
HAWKER BEECHCRAFT CORP.             7
BEECH AIRCRAFT CORPORATION          3
HAWKER BEECH                        3
Name: count, dtype: int64
-----
BOEING
Make
BO

In [12]:
#  Drop the one missing Make
df = df.dropna(subset=["Make"]).copy()

# Merge spelling variants of the same manufacturer
df["Make"] = df["Make"].replace({
    "AIR TRACTOR INC": "AIR TRACTOR",
    "CIRRUS DESIGN CORP": "CIRRUS",
    "DEHAVILLAND": "DE HAVILLAND",
    "GRUMMAN AMERICAN": "GRUMMAN",
})

# Keep only makes with at least 50 accidents
counts = df["Make"].value_counts()
df = df[df["Make"].isin(counts[counts >= 50].index)].copy()

print(df.shape)
print("Makes remaining:", df["Make"].nunique())

(55102, 36)
Makes remaining: 52


Assumptions:
Dropped 1 row with a missing Make
i kept makes with at least 50 accidents since the rate from small groups is not statistically reliable
i kept those that are of different companies and product lines

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [13]:
print("Missing Model:", df["Model"].isna().sum())

# Is the same Model label used by more than one Make?
makes_per_model = df.groupby("Model")["Make"].nunique()
print("Models used by more than one Make:", (makes_per_model > 1).sum())
print(makes_per_model[makes_per_model > 1].sort_values(ascending=False).head(15))

Missing Model: 12
Models used by more than one Make: 106
Model
500         4
G-164B      4
S2R         4
G-164A      4
690A        3
7AC         3
112A        3
7EC         3
7ECA        3
112         3
8GCBC       3
8KCAB       3
S-2R        3
G-164C      3
NAVION A    3
Name: Make, dtype: int64


In [14]:
# Remove rows with a missing Model
df = df.dropna(subset=["Model"]).copy()

#Derived column: unique identifier for a plane type
df["Make.Model"] = df["Make"] + " " + df["Model"]

print(df.shape)
print("Unique Make.Model:", df["Make.Model"].nunique())
print(df["Make.Model"].value_counts().head(15))

(55090, 37)
Unique Make.Model: 2238
Make.Model
CESSNA 152         2210
CESSNA 172         1613
CESSNA 172N        1080
PIPER PA-28-140     857
CESSNA 172M         755
CESSNA 150          742
CESSNA 172P         661
CESSNA 182          608
CESSNA 180          593
CESSNA 150M         550
PIPER PA-18-150     547
PIPER PA-18         546
PIPER PA-28-180     545
PIPER PA-28-161     519
PIPER PA-28-181     502
Name: count, dtype: int64


dropped 12 rows with missing model
created make.model as anique identifier for each plane type

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [15]:
cols = ["Engine.Type", "Weather.Condition", "Number.of.Engines",
        "Purpose.of.flight", "Broad.phase.of.flight"]

for c in cols:
    print(c)
    print(df[c].value_counts(dropna=False))
    print("-----")

Engine.Type
Engine.Type
Reciprocating      48799
NaN                 2795
Turbo Prop          2132
Turbo Fan            780
Unknown              383
Turbo Jet            188
Turbo Shaft           11
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64
-----
Weather.Condition
Weather.Condition
VMC    48533
IMC     4303
NaN     1638
UNK      485
Unk      131
Name: count, dtype: int64
-----
Number.of.Engines
Number.of.Engines
1.0    45913
2.0     6768
NaN     2084
3.0      139
4.0      107
0.0       79
Name: count, dtype: int64
-----
Purpose.of.flight
Purpose.of.flight
Personal                     32907
Instructional                 7769
Aerial Application            3080
Unknown                       3056
Business                      2654
NaN                           2373
Positioning                    884
Other Work Use                 487
Ferry                          474
Aerial Observation             365
Executive/corporate            269
Public Aircraft    

In [16]:
# Treat "unknown" labels as missing
df["Engine.Type"] = df["Engine.Type"].replace({"Unknown": np.nan, "UNK": np.nan})
df["Weather.Condition"] = df["Weather.Condition"].replace({"UNK": np.nan, "Unk": np.nan})
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].replace({"Unknown": np.nan})
df["Purpose.of.flight"] = df["Purpose.of.flight"].replace({"Unknown": np.nan})

# 0 engines is impossible for an airplane
df["Number.of.Engines"] = df["Number.of.Engines"].replace({0.0: np.nan})

# Merge duplicate labels in Purpose.of.flight
df["Purpose.of.flight"] = df["Purpose.of.flight"].replace({
    "Air Race/show": "Air Race show",
    "Public Aircraft - Federal": "Public Aircraft",
    "Public Aircraft - State": "Public Aircraft",
    "Public Aircraft - Local": "Public Aircraft",
})

for c in ["Engine.Type", "Weather.Condition", "Number.of.Engines",
          "Purpose.of.flight", "Broad.phase.of.flight"]:
    print(c, "missing:", df[c].isna().sum())

Engine.Type missing: 3179
Weather.Condition missing: 2254
Number.of.Engines missing: 2163
Purpose.of.flight missing: 5429
Broad.phase.of.flight missing: 15587


unknown and unk means the value was not recoreded so they were converted to NaN and NaNs was kept 
79 rows was set to NaN since 0 engines is not possible for an aeroplane

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [17]:
print((df.isna().mean() * 100).round(1).sort_values(ascending=False))

Schedule                  89.1
Air.carrier               83.4
FAR.Description           69.6
Latitude                  61.8
Longitude                 61.8
Airport.Code              38.8
Airport.Name              36.0
Broad.phase.of.flight     28.3
Publication.Date          16.9
Purpose.of.flight          9.9
Engine.Type                5.8
Report.Status              5.3
Weather.Condition          4.1
Number.of.Engines          3.9
Registration.Number        1.0
Country                    0.1
Location                   0.0
Event.Id                   0.0
Event.Date                 0.0
Accident.Number            0.0
Investigation.Type         0.0
Model                      0.0
Make                       0.0
Amateur.Built              0.0
Aircraft.Category          0.0
Aircraft.damage            0.0
Injury.Severity            0.0
Total.Serious.Injuries     0.0
Total.Fatal.Injuries       0.0
Total.Minor.Injuries       0.0
Total.Uninjured            0.0
Total.Occupants            0.0
FatalSer

In [18]:
cols_to_drop = ["Schedule", "Air.carrier", "FAR.Description",
                "Latitude", "Longitude",
                "Airport.Code", "Airport.Name",
                "Publication.Date"]
df = df.drop(columns=cols_to_drop)
print(df.shape)

(55090, 29)


columns with large share of missing values was dropped which includes scedule aircarrier ,far description,longitude and latitude
broad.phase.of flight was kept since it is one of the factors used for analyzing and that rows withNaN Will only be excluded when that particular factor is analyzed

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [19]:
df.to_csv("data/cleaned_aviation_data.csv", index=False)
print("Saved:", df.shape)

Saved: (55090, 29)


dropped columns with large share of missing values
cleaned data was saved to cleaned_aviation_data.csv for use in analysis